# Union-Find 演習 notebook：11.1〜11.4

この notebook は、Union-Find を使う代表的な 4 問を演習するためのものです。

扱う問題は次の 4 つです。

- 11.1 ABC 075 C - Bridge
- 11.2 ABC 120 D - Decayed Bridges
- 11.3 ABC 049 D - 連結 / Connectivity
- 11.4 ABC 087 D - People on a Line

各章は「プログラミングコンテスト風の問題文」「考え方」「Python 模範解答」「サンプルテスト」で構成しています。問題文は学習用に要点を保って書き直したものです。公式問題文そのものを再掲するものではありません。

参考リンク：

- ABC 075 C: https://atcoder.jp/contests/abc075/tasks/abc075_c
- ABC 120 D: https://atcoder.jp/contests/abc120/tasks/abc120_d
- ABC 049 D: https://atcoder.jp/contests/abc049/tasks/arc065_b
- ABC 087 D: https://atcoder.jp/contests/abc087/tasks/arc090_b


## notebook 用のテスト補助関数

各解答では、notebook 上で扱いやすいように `solve_xx(input_str: str) -> str` という形式で書いています。AtCoder に提出するときは、最後に `print(solve(sys.stdin.read()))` のように標準入力を渡してください。


In [1]:
def run_sample(func, sample_input: str, expected_output: str) -> None:
    """notebook 上でサンプルを簡単に確認するための関数。"""
    actual = func(sample_input).strip()
    expected = expected_output.strip()
    print("actual:")
    print(actual)
    print("expected:")
    print(expected)
    assert actual == expected, "出力が期待値と違います"
    print("OK")


# 11.1 ABC 075 C - Bridge

## 問題文

`N` 頂点 `M` 辺の無向連結グラフが与えられます。頂点番号は `1, 2, ..., N` です。`i` 番目の辺は頂点 `a_i` と頂点 `b_i` を結びます。

辺を 1 本取り除いたとき、グラフが連結でなくなるなら、その辺を「橋」と呼びます。与えられた `M` 本の辺のうち、橋であるものの個数を求めてください。

## 入力形式

```text
N M
a_1 b_1
a_2 b_2
...
a_M b_M
```

## 制約

- `2 <= N <= 50`
- `N-1 <= M <= min(N(N-1)/2, 50)`
- `1 <= a_i < b_i <= N`
- 自己ループと多重辺はない
- 最初のグラフは連結

## 出力形式

橋の本数を 1 行に出力してください。

## サンプル

入力：

```text
7 7
1 3
2 7
3 4
4 5
4 6
5 6
6 7
```

出力：

```text
4
```


## 考え方

`M <= 50` と小さいので、各辺を 1 本ずつ「使わない」と仮定して、残りの辺でグラフが連結かどうかを調べれば十分です。

各回で Union-Find を新しく作り、取り除く辺以外をすべて `union` します。その後、すべての頂点が頂点 `0` と同じ連結成分に入っているか確認します。連結でなければ、その取り除いた辺は橋です。

計算量はおおよそ `O(M(N+M) α(N))` です。この問題の制約では十分高速です。


In [2]:
class UnionFind:
    def __init__(self, n):
        self.parent = [-1] * n
        self.size_list = [1] * n

    # x の根を求める
    def find(self, x):
        if self.parent[x] == -1:
            return x

        self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    # x と y が同じグループか判定する
    def same(self, x, y):
        return self.find(x) == self.find(y)

    # x を含むグループと y を含むグループを併合する
    def union(self, x, y):
        x = self.find(x)
        y = self.find(y)

        if x == y:
            return False

        if self.size_list[x] < self.size_list[y]:
            x, y = y, x

        self.parent[y] = x
        self.size_list[x] += self.size_list[y]

        return True

    # x を含むグループのサイズを返す
    def size(self, x):
        return self.size_list[self.find(x)]


def solve_11_1(input_str: str) -> str:
    it = iter(input_str.split())
    N = int(next(it))
    M = int(next(it))
    edges = [(int(next(it)) - 1, int(next(it)) - 1) for _ in range(M)]

    answer = 0

    for skip in range(M):
        uf = UnionFind(N)

        for j, (a, b) in enumerate(edges):
            if j == skip:
                continue
            uf.union(a, b)

        connected = all(uf.same(0, v) for v in range(N))
        if not connected:
            answer += 1

    return str(answer)


In [3]:
sample_11_1 = """\
7 7
1 3
2 7
3 4
4 5
4 6
5 6
6 7
"""

expected_11_1 = """\
4
"""

run_sample(solve_11_1, sample_11_1, expected_11_1)


actual:
4
expected:
4
OK


AtCoder に提出する場合の末尾は、例えば次のようにします。

```python
import sys
print(solve_11_1(sys.stdin.read()))
```


# 11.2 ABC 120 D - Decayed Bridges

## 問題文

`N` 個の島と `M` 本の橋があります。`i` 番目の橋は島 `A_i` と島 `B_i` を双方向に結びます。最初は、すべての島の組について、橋をたどって行き来できます。

橋は老朽化により、`1` 番目、`2` 番目、...、`M` 番目の順に崩落します。

ある時点の「不便さ」を、行き来できない島の組 `(a, b)`、ただし `a < b`、の個数とします。各 `i = 1, 2, ..., M` について、`i` 番目の橋が崩落した直後の不便さを求めてください。

## 入力形式

```text
N M
A_1 B_1
A_2 B_2
...
A_M B_M
```

## 制約

- `2 <= N <= 100000`
- `1 <= M <= 100000`
- `1 <= A_i < B_i <= N`
- 同じ橋は 2 回以上現れない
- 最初の不便さは `0`

## 出力形式

`M` 行出力します。`i` 行目には、`i` 番目の橋が崩落した直後の不便さを出力してください。

## サンプル

入力：

```text
4 5
1 2
3 4
1 3
2 3
1 4
```

出力：

```text
0
0
4
5
6
```


## 考え方

辺を削除していく過程をそのまま追うのは難しいです。Union-Find は「つながっていない成分をつなぐ」のが得意ですが、「つながっている成分を切る」のは苦手だからです。

そこで、時間を逆向きに見ます。すべての橋が崩落した状態、つまり辺が 0 本の状態から、橋を `M, M-1, ..., 1` の順に追加していきます。

辺が 0 本のとき、行き来できない島の組はすべての組なので、初期値は `N(N-1)/2` です。橋を追加して異なる成分 `A` と `B` がつながると、その 2 成分間の `size(A) * size(B)` 組が行き来可能になるので、不便さからその分を引きます。

ポイントは、答えを記録してから辺を追加することです。逆順処理で、辺 `i` を追加する直前の状態は「前向きに見て、辺 `i` が崩落した直後」の状態に対応します。

計算量は `O((N+M) α(N))` です。


In [4]:
class UnionFind:
    def __init__(self, n):
        self.parent = [-1] * n
        self.size_list = [1] * n

    # x の根を求める
    def find(self, x):
        if self.parent[x] == -1:
            return x

        self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    # x と y が同じグループか判定する
    def same(self, x, y):
        return self.find(x) == self.find(y)

    # x を含むグループと y を含むグループを併合する
    def union(self, x, y):
        x = self.find(x)
        y = self.find(y)

        if x == y:
            return False

        if self.size_list[x] < self.size_list[y]:
            x, y = y, x

        self.parent[y] = x
        self.size_list[x] += self.size_list[y]

        return True

    # x を含むグループのサイズを返す
    def size(self, x):
        return self.size_list[self.find(x)]


def solve_11_2(input_str: str) -> str:
    it = iter(input_str.split())
    N = int(next(it))
    M = int(next(it))
    edges = [(int(next(it)) - 1, int(next(it)) - 1) for _ in range(M)]

    uf = UnionFind(N)
    inconvenience = N * (N - 1) // 2
    ans = [0] * M

    for i in range(M - 1, -1, -1):
        a, b = edges[i]

        # この時点は「i 番目の橋が崩落した直後」に対応する
        ans[i] = inconvenience

        if not uf.same(a, b):
            inconvenience -= uf.size(a) * uf.size(b)
            uf.union(a, b)

    return "\n".join(map(str, ans))


In [5]:
sample_11_2 = """\
4 5
1 2
3 4
1 3
2 3
1 4
"""

expected_11_2 = """\
0
0
4
5
6
"""

run_sample(solve_11_2, sample_11_2, expected_11_2)


actual:
0
0
4
5
6
expected:
0
0
4
5
6
OK


AtCoder に提出する場合の末尾は、例えば次のようにします。

```python
import sys
print(solve_11_2(sys.stdin.read()))
```


# 11.3 ABC 049 D - 連結 / Connectivity

## 問題文

`N` 個の都市があります。都市どうしを結ぶものとして、道路と鉄道の 2 種類があります。

道路は `K` 本あり、`i` 番目の道路は都市 `p_i` と都市 `q_i` を結びます。鉄道は `L` 本あり、`i` 番目の鉄道は都市 `r_i` と都市 `s_i` を結びます。

都市 `x` から都市 `y` へ、道路だけを使って移動できるとき、`x` と `y` は道路で連結であるとします。同様に、鉄道だけを使って移動できるとき、鉄道で連結であるとします。

各都市 `i` について、都市 `i` と「道路でも鉄道でも連結である」都市が何個あるかを求めてください。都市 `i` 自身も数えます。

## 入力形式

```text
N K L
p_1 q_1
...
p_K q_K
r_1 s_1
...
r_L s_L
```

## 制約

- `2 <= N <= 200000`
- `1 <= K, L <= 100000`
- 都市番号は `1` 以上 `N` 以下
- 各道路・各鉄道に重複はない

## 出力形式

`N` 個の整数を空白区切りで出力してください。`i` 番目の整数が都市 `i` の答えです。

## サンプル

入力：

```text
4 3 1
1 2
2 3
3 4
2 3
```

出力：

```text
1 2 2 1
```


## 考え方

道路の連結関係と鉄道の連結関係は別々に管理します。そこで、Union-Find を 2 つ用意します。

- `road_uf`: 道路だけで見た連結成分
- `rail_uf`: 鉄道だけで見た連結成分

ある都市 `i` の状態は、次のペアで表せます。

```python
(road_uf.find(i), rail_uf.find(i))
```

2 つの都市が道路でも鉄道でも連結であるためには、このペアが完全に一致する必要があります。よって、全都市についてペアを作り、同じペアが何回出現するかを数えれば、それが各都市の答えになります。

計算量は `O((N+K+L) α(N))` です。


In [6]:
from collections import Counter


class UnionFind:
    def __init__(self, n):
        self.parent = [-1] * n
        self.size_list = [1] * n

    # x の根を求める
    def find(self, x):
        if self.parent[x] == -1:
            return x

        self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    # x と y が同じグループか判定する
    def same(self, x, y):
        return self.find(x) == self.find(y)

    # x を含むグループと y を含むグループを併合する
    def union(self, x, y):
        x = self.find(x)
        y = self.find(y)

        if x == y:
            return False

        if self.size_list[x] < self.size_list[y]:
            x, y = y, x

        self.parent[y] = x
        self.size_list[x] += self.size_list[y]

        return True

    # x を含むグループのサイズを返す
    def size(self, x):
        return self.size_list[self.find(x)]


def solve_11_3(input_str: str) -> str:
    it = iter(input_str.split())
    N = int(next(it))
    K = int(next(it))
    L = int(next(it))

    road_uf = UnionFind(N)
    rail_uf = UnionFind(N)

    for _ in range(K):
        p = int(next(it)) - 1
        q = int(next(it)) - 1
        road_uf.union(p, q)

    for _ in range(L):
        r = int(next(it)) - 1
        s = int(next(it)) - 1
        rail_uf.union(r, s)

    pairs = [(road_uf.find(i), rail_uf.find(i)) for i in range(N)]
    count = Counter(pairs)

    ans = [count[pair] for pair in pairs]
    return " ".join(map(str, ans))


In [7]:
sample_11_3 = """\
4 3 1
1 2
2 3
3 4
2 3
"""

expected_11_3 = """\
1 2 2 1
"""

run_sample(solve_11_3, sample_11_3, expected_11_3)


actual:
1 2 2 1
expected:
1 2 2 1
OK


AtCoder に提出する場合の末尾は、例えば次のようにします。

```python
import sys
print(solve_11_3(sys.stdin.read()))
```


# 11.4 ABC 087 D - People on a Line

## 問題文

`N` 人が数直線上に立っています。人 `i` の座標を `x_i` とします。

`M` 個の情報が与えられます。`i` 番目の情報は `(L_i, R_i, D_i)` で、人 `R_i` は人 `L_i` より右に `D_i` だけ離れている、つまり

```text
x_{R_i} - x_{L_i} = D_i
```

であることを意味します。

すべての情報を同時に満たすような座標 `x_1, x_2, ..., x_N` が存在するか判定してください。

## 入力形式

```text
N M
L_1 R_1 D_1
L_2 R_2 D_2
...
L_M R_M D_M
```

## 制約

- `1 <= N <= 100000`
- `0 <= M <= 200000`
- `1 <= L_i, R_i <= N`
- `0 <= D_i <= 10000`
- `L_i != R_i`
- 同じ unordered な組に対する情報は高々 1 回

## 出力形式

条件を満たす座標の割り当てが存在するなら `Yes`、存在しないなら `No` と出力してください。

## サンプル 1

入力：

```text
3 3
1 2 1
2 3 1
1 3 2
```

出力：

```text
Yes
```

## サンプル 2

入力：

```text
3 3
1 2 1
2 3 1
1 3 5
```

出力：

```text
No
```


## 考え方

通常の Union-Find は「同じ連結成分かどうか」だけを管理します。この問題では、それに加えて「同じ成分内で、座標差がいくつか」も管理する必要があります。

そこで重み付き Union-Find を使います。

この実装では、`weight(x)` を「`x` の親方向をたどったときの根からの相対値」として管理します。`diff(x, y)` は `weight(y) - weight(x)`、つまり `x` から見た `y` の相対位置になります。

情報 `(L, R, D)` は、`x_R - x_L = D` です。

- まだ別成分なら、この制約を満たすように成分を結合します。
- すでに同じ成分なら、既存の `diff(L, R)` が `D` と一致するか確認します。

矛盾が 1 つでもあれば `No`、最後まで矛盾しなければ `Yes` です。

計算量は `O((N+M) α(N))` です。


In [8]:
class WeightedUnionFind:
    """
    重み付き Union-Find。

    weight(x) は、x の所属する根を基準にした x の相対値を返す。
    diff(x, y) は weight(y) - weight(x) を返す。
    union(x, y, w) は「weight(y) - weight(x) = w」を追加する。
    """

    def __init__(self, n: int):
        self.parent = list(range(n))
        self.size_arr = [1] * n
        self.diff_weight = [0] * n

    def find(self, x: int) -> int:
        if self.parent[x] == x:
            return x
        p = self.parent[x]
        r = self.find(p)
        self.diff_weight[x] += self.diff_weight[p]
        self.parent[x] = r
        return r

    def weight(self, x: int) -> int:
        self.find(x)
        return self.diff_weight[x]

    def diff(self, x: int, y: int) -> int:
        return self.weight(y) - self.weight(x)

    def union(self, x: int, y: int, w: int) -> bool:
        """
        制約 weight(y) - weight(x) = w を追加する。
        矛盾していなければ True、矛盾していれば False を返す。
        """
        rx = self.find(x)
        ry = self.find(y)
        wx = self.weight(x)
        wy = self.weight(y)

        if rx == ry:
            return wy - wx == w

        if self.size_arr[rx] < self.size_arr[ry]:
            # rx を ry の子にする
            self.parent[rx] = ry
            self.diff_weight[rx] = wy - wx - w
            self.size_arr[ry] += self.size_arr[rx]
        else:
            # ry を rx の子にする
            self.parent[ry] = rx
            self.diff_weight[ry] = w + wx - wy
            self.size_arr[rx] += self.size_arr[ry]

        return True


def solve_11_4(input_str: str) -> str:
    it = iter(input_str.split())
    N = int(next(it))
    M = int(next(it))

    uf = WeightedUnionFind(N)

    for _ in range(M):
        L = int(next(it)) - 1
        R = int(next(it)) - 1
        D = int(next(it))

        if not uf.union(L, R, D):
            return "No"

    return "Yes"


In [9]:
sample_11_4_1 = """\
3 3
1 2 1
2 3 1
1 3 2
"""

expected_11_4_1 = """\
Yes
"""

run_sample(solve_11_4, sample_11_4_1, expected_11_4_1)


actual:
Yes
expected:
Yes
OK


In [10]:
sample_11_4_2 = """\
3 3
1 2 1
2 3 1
1 3 5
"""

expected_11_4_2 = """\
No
"""

run_sample(solve_11_4, sample_11_4_2, expected_11_4_2)


actual:
No
expected:
No
OK


AtCoder に提出する場合の末尾は、例えば次のようにします。

```python
import sys
print(solve_11_4(sys.stdin.read()))
```


# 追加演習のすすめ

余力があれば、次の順で復習すると効果的です。

1. 11.1 を、Union-Find ではなく DFS/BFS で連結判定して書く。
2. 11.2 で、なぜ「答えを記録してから辺を追加する」のかを小さい例で手計算する。
3. 11.3 で、`(道路の根, 鉄道の根)` のペアを紙に書き出す。
4. 11.4 の重み付き Union-Find で、`diff_weight` が何を表しているかを 3 頂点の例で追う。

Union-Find は「削除」より「追加」が得意です。削除が出てきたら、逆順にして追加問題へ変換できないかをまず疑うのがよいです。
